# audio2chordpro（Google Colab）

音源 + AMT の MIDI（コード・調・拍・歌メロ）+ 歌詞 → **歌詞の上にコードが載った ChordPro** を作ります。

1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選ぶ
2. 上のセルから順に実行する

- 初回はライブラリ（PyTorch など）とモデル（wav2vec2、demucs、SheetSage2）のダウンロードに数分かかります
- 歌メロの採譜に使う [SheetSage2](https://huggingface.co/m-a-p/SheetSage2) の重みは **CC BY-NC 4.0（非商用）** です。
  使わない場合は下の設定で `MELODY` を `amt`（MIDI の melody トラック）にしてください

## 1. セットアップ

In [ ]:
#@title リポジトリの取得と依存ライブラリのインストール
REPO_URL = "https://github.com/anime-song/audio2chordpro"  #@param {type:"string"}
REPO = "/content/audio2chordpro"

import os

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU がありません（CPU でも動きますが遅くなります）"
if not os.path.exists(REPO):
    !git clone --depth 1 {REPO_URL} {REPO}
!pip install -q uv
# uv.lock で固定したバージョンを、Colab の Python とは別の環境（REPO/.venv）に入れる
!cd {REPO} && uv sync --frozen

In [ ]:
#@title （任意）Google ドライブに中間結果を保存する
#@markdown 分離したボーカル・CTC・SheetSage2 の結果をドライブに置くと、次のセッションで同じ曲を再実行するときに速くなります。
USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_CACHE = "/content/drive/MyDrive/audio2chordpro/cache"  #@param {type:"string"}

CACHE_DIR = "/content/cache"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = DRIVE_CACHE
os.makedirs(CACHE_DIR, exist_ok=True)
print("中間結果の置き場所:", CACHE_DIR)

## 2. 入力

In [ ]:
#@title 音源・MIDI・歌詞をアップロード
#@markdown 音源（mp3 / wav / flac / m4a）と AMT の MIDI（.mid）、あれば歌詞（.txt、UTF-8）を選んでください。
from pathlib import Path
from google.colab import files

INPUT_DIR = Path("/content/input")
INPUT_DIR.mkdir(exist_ok=True)
for name, data in files.upload().items():
    (INPUT_DIR / name).write_bytes(data)


def latest(*exts):
    found = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in exts]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None


AUDIO = latest(".mp3", ".wav", ".flac", ".m4a", ".ogg")
MIDI = latest(".mid", ".midi")
LYRICS_FILE = latest(".txt")
print("音源  :", AUDIO)
print("MIDI  :", MIDI)
print("歌詞  :", LYRICS_FILE or "（なし。次のセルに貼り付けるか、コード譜だけを作る）")

In [ ]:
#@title （任意）歌詞を貼り付ける
#@markdown 歌詞のファイルをアップロードしなかった場合は、下の `LYRICS` に貼り付けてください。
#@markdown 1行 = ChordPro の1行、空行 = 段落の区切り。`漢字(かな)` でルビ（読み）を指定できます。
#@markdown 空のままなら歌詞なし（コード譜だけ）になります。
LYRICS = """
"""

if LYRICS.strip():
    LYRICS_FILE = INPUT_DIR / "lyrics.txt"
    LYRICS_FILE.write_text(LYRICS.strip() + "\n", encoding="utf-8")
print("歌詞  :", LYRICS_FILE or "なし（コード譜だけ）")

In [ ]:
#@title 曲の情報と設定
#@markdown 曲の情報（`{title}` と `{subtitle}` になります。空欄は省略）
TITLE = ""  #@param {type:"string"}
ARTIST = ""  #@param {type:"string"}
LYRICIST = ""  #@param {type:"string"}
COMPOSER = ""  #@param {type:"string"}
ARRANGER = ""  #@param {type:"string"}
#@markdown 設定
MELODY = "sheetsage"  #@param ["sheetsage", "amt"]
BEATS = "amt"  #@param ["amt", "sheetsage"]
SIMPLIFY = False  #@param {type:"boolean"}
HEAD_OUTSIDE = False  #@param {type:"boolean"}
TAIL_GRID = True  #@param {type:"boolean"}

## 3. 実行

In [ ]:
#@title ChordPro を作る
import shlex

assert AUDIO and MIDI, "音源と MIDI をアップロードしてください"
OUT = Path("/content/output") / f"{AUDIO.stem}.cho"
OUT.parent.mkdir(exist_ok=True)

cmd = ["uv", "run", "--frozen", "--project", REPO, "audio2chordpro", str(AUDIO), "--midi", str(MIDI),
       "-o", str(OUT), "--cache-dir", CACHE_DIR, "--melody", MELODY, "--beats", BEATS,
       "--save-alignment", str(OUT.with_suffix(".align.json"))]
if LYRICS_FILE:
    cmd += ["--lyrics", str(LYRICS_FILE)]
for opt, value in (("--title", TITLE), ("--artist", ARTIST), ("--lyricist", LYRICIST),
                   ("--composer", COMPOSER), ("--arranger", ARRANGER)):
    if value:
        cmd += [opt, value]
if SIMPLIFY:
    cmd.append("--simplify")
if HEAD_OUTSIDE:
    cmd.append("--head-outside")
if not TAIL_GRID:
    cmd.append("--no-tail-grid")

cmdline = shlex.join(cmd)
!{cmdline}
print(OUT.read_text(encoding="utf-8"))

In [ ]:
#@title 結果をダウンロード
files.download(str(OUT))